# ADP1 BERDL Fold Change Flux Analysis

## Objectives

This notebook performs flux analysis based on proteomics fold changes between strains,
using the **BERDL fitness-constrained flux** as the reference distribution (from ADP1BERDLFitnessFluxFitting).

### Analysis Goals:
1. Load proteomics data (log2 format) and average replicates
2. Load reference flux distribution from BERDL fitness-constrained pyruvate solution
3. For each strain (target condition), compute fold changes relative to ADP1 (reference)
4. Use MSExpression.fit_flux_to_proteomics_fold_change_data to fit fluxes
5. Analyze which protein fold changes could/could not be implemented
6. Compare flux changes across strains

### Strains (Target Conditions):
- **ACN2586**: Initial construct
- **ACN2821**: Evolved strain
- **ACN3425, ACN3427, ACN3429, ACN3430**: Additional strain variants

### Reference Condition:
- **ADP1**: Wild-type Acinetobacter baylyi ADP1

### Reference Flux:
- **BERDL fitness-constrained flux** on pyruvate media (from ADP1BERDLFitnessFluxFitting)
- This differs from ADP1FoldChangeAnalysis which uses the lactate condition from ADP1MutantPhenotypeAnalysis

## Load Proteomics Data and Average Replicates

This step:
1. Loads proteomics data from Excel spreadsheet (log2 format)
2. Identifies all strain conditions including ADP1 reference
3. Averages replicates while keeping data in log2 format
4. Saves averaged expression data for subsequent analysis

In [5]:
%run util.py

# Load proteomics data in log2 format
raw_expression = MSExpression.from_spreadsheet(
    filename="data/ASCR_UGA_Proteomics_DgoA_add_strains_2025_pyruvate.xlsx",
    sheet_name="Imputed",
    skiprows=0,
    type="Log2",
    id_column="ACIAD"
)

# Define all strain conditions (including ADP1 reference)
all_strains = [
    "Pyruvate_ACN2586_DgoA_2025",
    "Pyruvate_ACN2821_DgoA_2025",
    "Pyruvate_ACN3425_DgoA_2025",
    "Pyruvate_ACN3427_DgoA_2025",
    "Pyruvate_ACN3429_DgoA_2025",
    "Pyruvate_ACN3430_DgoA_2025",
    "Pyruvate_ADP1_DgoA_2025"
]

# Reference condition (ADP1 wild-type)
reference_condition = "Pyruvate_ADP1_DgoA_2025"

# Target conditions (all strains except ADP1)
target_conditions = [s for s in all_strains if s != reference_condition]

print(f"Reference condition: {reference_condition}")
print(f"Target conditions: {target_conditions}")
print(f"Total features (genes): {len(raw_expression.features)}")

# Average replicates while keeping log2 format
averaged_expression = raw_expression.average_expression_replicates(all_strains)

print(f"\nAfter averaging replicates:")
print(f"  Conditions: {[c.id for c in averaged_expression.conditions]}")
print(f"  Data type: {averaged_expression.type}")

# Save strain info and averaged data
util.save("berdl_fc_strains", {
    "all_strains": all_strains,
    "reference_condition": reference_condition,
    "target_conditions": target_conditions
})
util.save("berdl_fc_averaged_expression", averaged_expression._data.to_dict())

print("\nSaved strain info and averaged expression data to datacache/")

2026-02-21 20:24:58,341 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-02-21 20:24:58,342 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-02-21 20:24:58,344 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-21 20:24:58,985 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-02-21 20:24:58,985 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-02-21 20:24:58,986 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-21 20:24:58,986 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-02-21 20:24:59,002 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo
/home/chenry/MyEnvs/modelseed_cplex/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


Reference condition: Pyruvate_ADP1_DgoA_2025
Target conditions: ['Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025']
Total features (genes): 2383

After averaging replicates:
  Conditions: ['Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025', 'Pyruvate_ADP1_DgoA_2025']
  Data type: Log2

Saved strain info and averaged expression data to datacache/


## Gene-Level Proteomics Fold Changes

This step:
1. Computes fold changes for each gene relative to ADP1 wild-type reference
2. Fold change = 2^(target_log2 - reference_log2), log2 FC = target_log2 - reference_log2
3. Saves six separate datacache files, one per strain: `berdl_fc_<strain>_vs_ADP1`
4. Each file is a dictionary: {gene_id: {"log2_fc": ..., "fold_change": ...}}

In [ ]:
%run util.py

import numpy as np
import pandas as pd

# Load averaged expression data
averaged_expression_data = util.load("berdl_fc_averaged_expression")
strain_info = util.load("berdl_fc_strains")

# Build DataFrame from saved data
expr_df = pd.DataFrame.from_dict(averaged_expression_data)

reference_condition = strain_info["reference_condition"]
target_conditions = strain_info["target_conditions"]

# Compute and save gene-level fold changes for each strain vs ADP1
for target in target_conditions:
    short_name = target.replace('Pyruvate_', '').replace('_DgoA_2025', '')
    
    strain_fc = {}
    for gene_id in expr_df.index:
        ref_val = expr_df.loc[gene_id, reference_condition]
        target_val = expr_df.loc[gene_id, target]
        
        # Both values are log2; fold change = 2^(target - reference)
        if pd.notna(ref_val) and pd.notna(target_val):
            log2_fc = target_val - ref_val
            linear_fc = 2 ** log2_fc
            strain_fc[gene_id] = round(float(linear_fc), 4)
    
    # Save individual file per strain
    cache_key = f"berdl_fc_{short_name}_vs_ADP1"
    util.save(cache_key, strain_fc)
    
    # Summary stats
    values = list(strain_fc.values())
    up_2x = sum(1 for v in values if v >= 2.0)
    down_2x = sum(1 for v in values if v <= 0.5)
    
    print(f"{short_name}: {len(strain_fc)} genes -> saved as '{cache_key}'")
    print(f"  Mean FC: {np.mean(values):.3f}, Median: {np.median(values):.3f}")
    print(f"  Up >=2x: {up_2x}, Down <=0.5x: {down_2x}")

print(f"\nSaved 6 separate fold change files to datacache/")

## Load BERDL Fitness-Constrained Reference Flux

This step:
1. Loads the BERDL fitness-constrained flux result from ADP1BERDLFitnessFluxFitting
2. Extracts the flux distribution to use as the reference baseline
3. This flux was produced by fitting to TnSeq fitness + essentiality data on pyruvate media
4. The reference flux will be scaled by proteomics fold changes to compute target fluxes

In [6]:
%run util.py

# Load the BERDL fitness-constrained result from the fitness flux fitting notebook
constrained_result = util.load("berdl_constrained_pyruvate_result", notebook_name="ADP1BERDLFitnessFluxFitting")

# Extract the constrained flux as our reference
reference_flux = constrained_result["fluxes"]

print(f"Loaded BERDL fitness-constrained reference flux:")
print(f"  Total reactions with flux data: {len(reference_flux)}")
print(f"  Non-zero fluxes: {sum(1 for v in reference_flux.values() if abs(v) > 1e-9)}")
print(f"  Growth rate: {constrained_result.get('growth_rate', 'N/A')}")
print(f"  Solution status: {constrained_result.get('status', 'N/A')}")

# Show classification summary from fitness fitting
print(f"\nFitness fitting classification:")
print(f"  On-On (active, expected active): {len(constrained_result.get('on_on', []))}")
print(f"  On-Off (inactive, expected active): {len(constrained_result.get('on_off', []))}")
print(f"  Off-On (active, expected inactive): {len(constrained_result.get('off_on', []))}")
print(f"  Off-Off (inactive, expected inactive): {len(constrained_result.get('off_off', []))}")

# Show some example fluxes
print(f"\nSample fluxes (first 10 non-zero):")
nonzero_count = 0
for rxn_id, flux in reference_flux.items():
    if abs(flux) > 1e-9:
        print(f"  {rxn_id}: {flux:.6f}")
        nonzero_count += 1
        if nonzero_count >= 10:
            break

# Save reference flux for later use
util.save("berdl_fc_reference_flux", reference_flux)

print("\nSaved BERDL reference flux to datacache/")

2026-02-21 20:25:12,184 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-02-21 20:25:12,186 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-02-21 20:25:12,187 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-21 20:25:12,828 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-02-21 20:25:12,829 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-02-21 20:25:12,830 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-21 20:25:12,831 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-02-21 20:25:12,846 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


Loaded BERDL fitness-constrained reference flux:
  Total reactions with flux data: 1504
  Non-zero fluxes: 421
  Growth rate: 0.10477394765814872
  Solution status: optimal

Fitness fitting classification:
  On-On (active, expected active): 347
  On-Off (inactive, expected active): 962
  Off-On (active, expected inactive): 0
  Off-Off (inactive, expected inactive): 195

Sample fluxes (first 10 non-zero):
  rxn00423_c0: 0.008371
  rxn00364_c0: 0.058112
  rxn03408_c0: 0.003361
  rxn01673_c0: 0.000659
  rxn05625_c0: -0.004016
  rxn04070_c0: 0.000168
  rxn00199_c0: 0.092602
  rxn00172_c0: -3.401461
  rxn00800_c0: 0.021191
  rxn10336_c0: 0.002599

Saved BERDL reference flux to datacache/


## Run Fold Change Flux Analysis for All Conditions

This step:
1. Caps reference flux at +/-20 to prevent extreme target fluxes
2. For each target condition (strain), computes fold changes relative to ADP1
3. Uses fit_flux_to_proteomics_fold_change_data to fit fluxes (fold changes capped at 3x up/down internally)
4. Uses the BERDL fitness-constrained flux as the reference distribution
5. Collects results including:
   - Fitted flux distributions
   - Reactions with significant flux changes
   - Proteins whose fold changes could/could not be implemented
   - Reaction matching statistics

In [1]:
%run util.py
import pandas as pd
import cobra.io
import numpy as np

# Ceilings for fold change and flux
MAX_FOLD_CHANGE = 3.0       # 3x up or down
MAX_FLUX = 20.0             # Cap reference flux magnitude
LOG2_FC_CAP = np.log2(MAX_FOLD_CHANGE)  # ~1.585 log2 units

# Load saved data
strain_info = util.load("berdl_fc_strains")
averaged_expression_data = util.load("berdl_fc_averaged_expression")
reference_flux_raw = util.load("berdl_fc_reference_flux")

# Cap reference flux at +/-MAX_FLUX
reference_flux = {}
capped_count = 0
for rxn_id, flux in reference_flux_raw.items():
    if flux > MAX_FLUX:
        reference_flux[rxn_id] = MAX_FLUX
        capped_count += 1
    elif flux < -MAX_FLUX:
        reference_flux[rxn_id] = -MAX_FLUX
        capped_count += 1
    else:
        reference_flux[rxn_id] = flux

print(f"Reference flux: {capped_count} reactions capped to +/-{MAX_FLUX}")

# Load model
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")
model.model.reactions.get_by_id("rxn01332_c0").lower_bound = 0
model.model.reactions.get_by_id("rxn01332_c0").upper_bound = 0
model.model.reactions.get_by_id("DgoA").lower_bound = -1000
model.model.reactions.get_by_id("DgoA").upper_bound = 1000

pyruvate_media = util.get_media("KBaseMedia/Carbon-Pyruvic-Acid", msmedia=True)
util.set_media(model, pyruvate_media)

# Rebuild MSExpression from saved data
expression_df = pd.DataFrame.from_dict(averaged_expression_data)
expression_df = expression_df.reset_index()
expression_df = expression_df.rename(columns={'index': 'gene_id'})

# Cap log2 fold changes in expression data to enforce 3x ceiling
# Compute per-gene log2 FC = target - reference, then clamp
ref_col = strain_info["reference_condition"]
target_cols = strain_info["target_conditions"]
genes_capped = 0
for col in target_cols:
    if col in expression_df.columns and ref_col in expression_df.columns:
        log2_fc = expression_df[col] - expression_df[ref_col]
        # Clamp: where |log2_fc| > LOG2_FC_CAP, adjust target value
        too_high = log2_fc > LOG2_FC_CAP
        too_low = log2_fc < -LOG2_FC_CAP
        genes_capped += too_high.sum() + too_low.sum()
        expression_df.loc[too_high, col] = expression_df.loc[too_high, ref_col] + LOG2_FC_CAP
        expression_df.loc[too_low, col] = expression_df.loc[too_low, ref_col] - LOG2_FC_CAP

print(f"Expression data: {genes_capped} gene-condition pairs capped to +/-{LOG2_FC_CAP:.3f} log2 ({MAX_FOLD_CHANGE}x)")

print(f"Expression DataFrame shape: {expression_df.shape}")
print(f"Columns: {list(expression_df.columns)}")

# Create MSExpression from DataFrame
expression = MSExpression.from_dataframe(
    genome_or_model=util.get_msgenome_from_dict(util.load("ADP1Genome")["data"]),
    df=expression_df,
    id_column='gene_id',
    type="Log2",
    create_missing_features=True
)

print(f"MSExpression created with {len(expression.features)} features and {len(expression.conditions)} conditions")
print(f"Conditions: {[c.id for c in expression.conditions]}")

reference_condition = strain_info["reference_condition"]
target_conditions = strain_info["target_conditions"]

print(f"\nReference condition: {reference_condition}")
print(f"Reference flux source: BERDL fitness-constrained (pyruvate)")
print(f"Max fold change: {MAX_FOLD_CHANGE}x, Max flux: {MAX_FLUX}")
print(f"Analyzing {len(target_conditions)} target conditions")
print("=" * 80)

# Store all results
fold_change_results = {}

for target_condition in target_conditions:
    print(f"\nProcessing: {target_condition}")
    print("-" * 60)
    
    try:
        # Create a fresh model copy for each condition
        model_copy = MSModelUtil.from_cobrapy(cobra.io.json.to_json(model.model))
        
        # Run fold change flux fitting
        # quadratic_formulation=True requires CPLEX/Gurobi (not GLPK)
        result = expression.fit_flux_to_proteomics_fold_change_data(
            model=model_copy,
            reference_condition=reference_condition,
            target_condition=target_condition,
            reference_flux=reference_flux,
            zero_flux=0.001,
            least_squares=True,
            fold_change_thresholds=[1.0, 2.0, 3.0],
            quadratic_formulation=True
        )
        
        # Extract key statistics
        stats = result['statistics']
        print(f"  Reactions with fold change data: {stats['total_reactions_with_fold_change']}")
        print(f"  Genes with fold change data: {stats['total_genes_with_fold_change']}")
        print(f"  Reactions matched: {stats['reactions_matched']}")
        print(f"  Reactions not matched: {stats['reactions_not_matched']}")
        print(f"  Proteins implemented: {stats['proteins_implemented']}")
        print(f"  Proteins not implemented: {stats['proteins_not_implemented']}")
        
        # Store result
        fold_change_results[target_condition] = {
            'fluxes': result['solution'].fluxes.to_dict() if result['solution'] else {},
            'objective_value': result['solution'].objective_value if result['solution'] else None,
            'status': result['solution'].status if result['solution'] else 'error',
            'target_flux': result['target_flux'],
            'fold_changes': {k: v for k, v in result['fold_changes'].items()},
            'gene_fold_changes': result['gene_fold_changes'],
            'flux_changes': {
                'increased_1std': [r['rxn_id'] for r in result['flux_changes']['increased_1std']],
                'increased_2std': [r['rxn_id'] for r in result['flux_changes']['increased_2std']],
                'increased_3std': [r['rxn_id'] for r in result['flux_changes']['increased_3std']],
                'decreased_1std': [r['rxn_id'] for r in result['flux_changes']['decreased_1std']],
                'decreased_2std': [r['rxn_id'] for r in result['flux_changes']['decreased_2std']],
                'decreased_3std': [r['rxn_id'] for r in result['flux_changes']['decreased_3std']]
            },
            'protein_changes': result['protein_changes'],
            'reaction_matching': result['reaction_matching'],
            'statistics': result['statistics']
        }
        
        print(f"  Status: SUCCESS")
        
    except Exception as e:
        print(f"  ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        fold_change_results[target_condition] = {
            'status': 'error',
            'error': str(e)
        }

# Save all results
util.save("ADP1BERDLFoldChangeAnalysis", fold_change_results)

print("\n" + "=" * 80)
print(f"Completed analysis for {len(fold_change_results)} conditions")
print("Results saved to datacache/ADP1BERDLFoldChangeAnalysis.json")

/home/chenry/Dropbox/Projects/KBUtilLib/src


[KBUtilLib] Failed to import rcsb_pdb_utils: ModuleNotFoundError: No module named 'aiohttp'


modelseedpy 0.4.2


2026-02-21 20:26:34,535 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-02-21 20:26:34,536 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token


loading biochemistry database from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


2026-02-21 20:26:42,609 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase
2026-02-21 20:26:43,254 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-02-21 20:26:43,255 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-02-21 20:26:43,256 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-21 20:26:43,257 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-02-21 20:26:43,309 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


cobrakbase 0.4.0
Reference flux: 0 reactions capped to +/-20.0


2026-02-21 20:26:44,131 - __main__.NotebookUtil - INFO - File not found in /home/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1BERDLFoldChangeAnalysis, loading from base datacache


Expression data: 733 gene-condition pairs capped to +/-1.585 log2 (3.0x)
Expression DataFrame shape: (2383, 8)
Columns: ['gene_id', 'Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025', 'Pyruvate_ADP1_DgoA_2025']
MSExpression created with 2383 features and 7 conditions
Conditions: ['Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025', 'Pyruvate_ADP1_DgoA_2025']

Reference condition: Pyruvate_ADP1_DgoA_2025
Reference flux source: BERDL fitness-constrained (pyruvate)
Max fold change: 3.0x, Max flux: 20.0
Analyzing 6 target conditions

Processing: Pyruvate_ACN2586_DgoA_2025
------------------------------------------------------------
  Reactions with fold change data: 1132
  Genes with fold change data: 2383
  Reactions matched: 636
  Reacti

## Summary Statistics

This step:
1. Loads the fold change analysis results
2. Generates summary tables comparing all conditions
3. Identifies common patterns across strains

In [2]:
%run util.py
import pandas as pd

# Load results
results = util.load("ADP1BERDLFoldChangeAnalysis")

print("BERDL FOLD CHANGE ANALYSIS SUMMARY")
print("Reference flux: BERDL fitness-constrained (pyruvate)")
print("=" * 100)

# Build summary table
summary_data = []
for condition, data in results.items():
    if data.get('status') == 'error':
        summary_data.append({
            'Condition': condition,
            'Status': 'ERROR',
            'Rxns w/ FC': 'N/A',
            'Genes w/ FC': 'N/A',
            'Matched': 'N/A',
            'Not Matched': 'N/A',
            'Proteins Impl': 'N/A',
            'Proteins Not Impl': 'N/A',
            'Inc 2x': 'N/A',
            'Dec 2x': 'N/A'
        })
    else:
        stats = data.get('statistics', {})
        fc = data.get('flux_changes', {})
        summary_data.append({
            'Condition': condition.replace('Pyruvate_', '').replace('_DgoA_2025', ''),
            'Status': data.get('status', 'unknown'),
            'Rxns w/ FC': stats.get('total_reactions_with_fold_change', 0),
            'Genes w/ FC': stats.get('total_genes_with_fold_change', 0),
            'Matched': stats.get('reactions_matched', 0),
            'Not Matched': stats.get('reactions_not_matched', 0),
            'Proteins Impl': stats.get('proteins_implemented', 0),
            'Proteins Not Impl': stats.get('proteins_not_implemented', 0),
            'Inc 2x': len(fc.get('increased_1std', [])),
            'Dec 2x': len(fc.get('decreased_1std', []))
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Save summary to Excel
os.makedirs(util.output_dir, exist_ok=True)
summary_df.to_excel(f"{util.output_dir}/berdl_fold_change_summary.xlsx", index=False)
print(f"\nSummary saved to {util.output_dir}/berdl_fold_change_summary.xlsx")

2026-02-21 20:27:16,528 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-02-21 20:27:16,529 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-02-21 20:27:16,530 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-21 20:27:17,161 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-02-21 20:27:17,162 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-02-21 20:27:17,163 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-21 20:27:17,164 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-02-21 20:27:17,180 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


BERDL FOLD CHANGE ANALYSIS SUMMARY
Reference flux: BERDL fitness-constrained (pyruvate)
Condition  Status  Rxns w/ FC  Genes w/ FC  Matched  Not Matched  Proteins Impl  Proteins Not Impl  Inc 2x  Dec 2x
  ACN2586 optimal        1132         2383      636          496             29                431     129      34
  ACN2821 optimal        1132         2383      810          322             18                292     127     274
  ACN3425 optimal        1132         2383      738          394             22                282     128     247
  ACN3427 optimal        1132         2383      718          414             27                360     117     253
  ACN3429 optimal        1132         2383      812          320             14                267     125      15
  ACN3430 optimal        1132         2383      747          385             18                304     101     234

Summary saved to /home/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ADP1BERDLFoldChangeAnalysi

## Protein Implementation Analysis

This step:
1. Identifies proteins with significant fold changes that could not be implemented
2. Analyzes why certain fold changes couldn't be matched in the flux solution
3. Identifies common bottlenecks across strains

In [3]:
%run util.py

# Load results
results = util.load("ADP1BERDLFoldChangeAnalysis")

print("PROTEINS WITH SIGNIFICANT FOLD CHANGES NOT IMPLEMENTED")
print("Reference flux: BERDL fitness-constrained (pyruvate)")
print("=" * 100)

# Collect proteins not implemented across all conditions
all_not_implemented = {}

for condition, data in results.items():
    if data.get('status') == 'error':
        continue
    
    short_name = condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')
    not_impl = data.get('protein_changes', {}).get('significant_not_implemented', {})
    
    print(f"\n{short_name}: {len(not_impl)} proteins not implemented")
    
    # Show top 10 by fold change magnitude
    sorted_proteins = sorted(not_impl.items(), key=lambda x: abs(x[1].get('log2_fc', 0)), reverse=True)
    
    for gene_id, info in sorted_proteins[:10]:
        log2_fc = info.get('log2_fc', 0)
        reason = info.get('reason', 'unknown')
        direction = 'UP' if log2_fc > 0 else 'DOWN'
        print(f"  {gene_id}: {direction} {abs(log2_fc):.2f} log2FC - {reason}")
        
        # Track across conditions
        if gene_id not in all_not_implemented:
            all_not_implemented[gene_id] = []
        all_not_implemented[gene_id].append({
            'condition': short_name,
            'log2_fc': log2_fc,
            'reason': reason
        })

# Find proteins consistently not implemented
print("\n" + "=" * 100)
print("PROTEINS NOT IMPLEMENTED IN MULTIPLE CONDITIONS")
print("=" * 100)

for gene_id, occurrences in sorted(all_not_implemented.items(), key=lambda x: len(x[1]), reverse=True):
    if len(occurrences) >= 3:
        conditions = [o['condition'] for o in occurrences]
        avg_fc = sum(o['log2_fc'] for o in occurrences) / len(occurrences)
        print(f"{gene_id}: Not implemented in {len(occurrences)} conditions, avg log2FC={avg_fc:.2f}")
        print(f"  Conditions: {', '.join(conditions)}")

2026-02-21 20:27:26,753 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-02-21 20:27:26,754 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-02-21 20:27:26,755 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-21 20:27:27,389 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-02-21 20:27:27,391 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-02-21 20:27:27,391 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-21 20:27:27,392 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-02-21 20:27:27,408 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


PROTEINS WITH SIGNIFICANT FOLD CHANGES NOT IMPLEMENTED
Reference flux: BERDL fitness-constrained (pyruvate)

ACN2586: 431 proteins not implemented
  DgoA: UP 1.58 log2FC - flux_could_not_match
  ACIAD_RS02925: DOWN 1.58 log2FC - no_reactions_in_model
  ACIAD_RS02525: UP 1.58 log2FC - no_reactions_in_model
  ACIAD_RS02480: DOWN 1.58 log2FC - flux_could_not_match
  ACIAD_RS02130: DOWN 1.58 log2FC - no_reactions_in_model
  ACIAD_RS01780: UP 1.58 log2FC - no_reactions_in_model
  ACIAD_RS01595: DOWN 1.58 log2FC - no_reactions_in_model
  ACIAD_RS04600: DOWN 1.58 log2FC - no_reactions_in_model
  ACIAD_RS04345: UP 1.58 log2FC - no_reactions_in_model
  ACIAD_RS04090: UP 1.58 log2FC - no_reactions_in_model

ACN2821: 292 proteins not implemented
  DgoA: UP 1.58 log2FC - flux_could_not_match
  ACIAD_RS17085: DOWN 1.58 log2FC - no_reactions_in_model
  ACIAD_RS02130: DOWN 1.58 log2FC - no_reactions_in_model
  ACIAD_RS02115: DOWN 1.58 log2FC - no_reactions_in_model
  ACIAD_RS01780: DOWN 1.58 log2FC -

## Compare with Lactate-Based Fold Change Analysis

This step:
1. Loads results from both BERDL and lactate-based fold change analyses
2. Compares key statistics (matched reactions, proteins implemented, etc.)
3. Identifies how the different reference flux affects the fold change fitting
4. Correlates fitted fluxes between the two reference flux approaches

In [ ]:
%run util.py
import pandas as pd
import numpy as np

# Load both sets of results
berdl_results = util.load("ADP1BERDLFoldChangeAnalysis")
lactate_results = util.load("ADP1FoldChangeAnalysis", notebook_name="ADP1FoldChangeAnalysis")

if lactate_results is None:
    print("WARNING: Could not load lactate-based fold change results.")
    print("Run ADP1FoldChangeAnalysis notebook first for comparison.")
else:
    print("COMPARISON: BERDL vs LACTATE REFERENCE FLUX")
    print("=" * 110)
    
    # Build comparison table
    comparison_data = []
    for condition in berdl_results.keys():
        short_name = condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')
        
        b_data = berdl_results[condition]
        l_data = lactate_results.get(condition, {})
        
        if b_data.get('status') == 'error' or l_data.get('status') == 'error':
            continue
        
        b_stats = b_data.get('statistics', {})
        l_stats = l_data.get('statistics', {})
        
        comparison_data.append({
            'Condition': short_name,
            'BERDL Matched': b_stats.get('reactions_matched', 0),
            'Lactate Matched': l_stats.get('reactions_matched', 0),
            'BERDL Prot Impl': b_stats.get('proteins_implemented', 0),
            'Lactate Prot Impl': l_stats.get('proteins_implemented', 0),
            'BERDL Not Impl': b_stats.get('proteins_not_implemented', 0),
            'Lactate Not Impl': l_stats.get('proteins_not_implemented', 0),
        })
    
    if comparison_data:
        comp_df = pd.DataFrame(comparison_data)
        print(comp_df.to_string(index=False))
        
        # Correlate fitted fluxes between the two approaches for each condition
        print("\n" + "=" * 110)
        print("FLUX CORRELATION: BERDL-fitted vs Lactate-fitted")
        print("-" * 70)
        
        for condition in berdl_results.keys():
            short_name = condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')
            
            b_data = berdl_results[condition]
            l_data = lactate_results.get(condition, {})
            
            if b_data.get('status') == 'error' or l_data.get('status') == 'error':
                continue
            
            b_fluxes = b_data.get('fluxes', {})
            l_fluxes = l_data.get('fluxes', {})
            
            # Find common reactions
            common = sorted(set(b_fluxes.keys()) & set(l_fluxes.keys()))
            if len(common) > 0:
                b_vals = np.array([b_fluxes[r] for r in common])
                l_vals = np.array([l_fluxes[r] for r in common])
                
                from scipy import stats as scipy_stats
                r, p = scipy_stats.pearsonr(b_vals, l_vals)
                rho, sp = scipy_stats.spearmanr(b_vals, l_vals)
                print(f"  {short_name}: Pearson r={r:.4f} (p={p:.2e}), Spearman rho={rho:.4f}, n={len(common)} rxns")
        
        # Save comparison
        os.makedirs(util.output_dir, exist_ok=True)
        comp_df.to_excel(f"{util.output_dir}/berdl_vs_lactate_comparison.xlsx", index=False)
        print(f"\nComparison saved to {util.output_dir}/berdl_vs_lactate_comparison.xlsx")

## Visualize Reference Flux on Escher Map

This step:
1. Loads the BERDL fitness-constrained reference flux
2. Visualizes it on the full.json Escher map
3. Generates an interactive HTML file for exploring the flux distribution

In [ ]:
%run util.py

# Load reference flux
reference_flux = util.load("berdl_fc_reference_flux")

# Load the model for metadata
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")

print(f"BERDL reference flux has {len(reference_flux)} reactions")
print(f"Non-zero fluxes: {sum(1 for v in reference_flux.values() if abs(v) > 1e-9)}")

# Create Escher map visualization for reference flux
os.makedirs(util.output_dir, exist_ok=True)
output_file = util.create_map_html2(
    model=model.model,
    flux=reference_flux,
    map="full",
    output_path=f"{util.output_dir}/escher_berdl_reference_flux.html",
)

print(f"\nReference flux map saved to: {output_file}")

# Display link to open the map
from IPython.display import HTML, display
display(HTML(f'<a href="{output_file}" target="_blank">Open BERDL Reference Flux Map</a>'))

## Visualize Fitted Flux for Selected Condition

This step:
1. Allows selection of a target condition to visualize
2. Loads the fitted flux from fold change analysis
3. Displays on Escher map to compare with reference

In [6]:
%run util.py

# Load fold change results
results = util.load("ADP1BERDLFoldChangeAnalysis")

# Select condition to visualize (change this to view different conditions)
selected_condition = "Pyruvate_ACN2586_DgoA_2025"

# Get short name for display
short_name = selected_condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')

print(f"Selected condition: {short_name}")
print(f"Available conditions: {list(results.keys())}")

# Get fitted flux for selected condition
condition_data = results[selected_condition]
if condition_data.get('status') == 'error':
    print(f"ERROR: {condition_data.get('error', 'Unknown error')}")
else:
    fitted_flux = condition_data['fluxes']
    
    print(f"\nFitted flux has {len(fitted_flux)} reactions")
    print(f"Non-zero fluxes: {sum(1 for v in fitted_flux.values() if abs(v) > 1e-9)}")
    
    # Load model for metadata
    model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")
    
    # Create Escher map visualization for fitted flux
    os.makedirs(util.output_dir, exist_ok=True)
    output_file = util.create_map_html2(
        model=model.model,
        flux=fitted_flux,
        map="full",
        output_path=f"{util.output_dir}/escher_berdl_fitted_flux_{short_name}.html",
    )
    
    print(f"\nFitted flux map saved to: {output_file}")
    
    # Display link
    from IPython.display import HTML, display
    display(HTML(f'<a href="{output_file}" target="_blank">Open Fitted Flux Map ({short_name})</a>'))

2026-02-21 18:20:28,707 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-02-21 18:20:28,708 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-02-21 18:20:28,709 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-21 18:20:29,338 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-02-21 18:20:29,340 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-02-21 18:20:29,341 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-21 18:20:29,341 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-02-21 18:20:29,357 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


Selected condition: ACN2586
Available conditions: ['Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025']

Fitted flux has 1504 reactions
Non-zero fluxes: 602


2026-02-21 18:20:30,123 - __main__.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-02-21 18:20:30,129 - __main__.NotebookUtil - INFO - Updated names for 759 reactions in map



Fitted flux map saved to: /home/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_fitted_flux_ACN2586.html


## Visualize Flux Differential (Fitted - Reference)

This step:
1. Computes the flux differential between fitted and BERDL reference fluxes
2. Visualizes on Escher map using directional color scheme
3. Positive values (green) = increased flux, Negative values (red) = decreased flux

In [6]:
%run util.py

# Load data
results = util.load("ADP1BERDLFoldChangeAnalysis")
reference_flux = util.load("berdl_fc_reference_flux")

# Select condition to compare (change this to view different conditions)
selected_condition = "Pyruvate_ACN2586_DgoA_2025"

# Get short name for display
short_name = selected_condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')

print(f"Computing flux differential for: {short_name}")

# Get fitted flux for selected condition
condition_data = results[selected_condition]
if condition_data.get('status') == 'error':
    print(f"ERROR: {condition_data.get('error', 'Unknown error')}")
else:
    fitted_flux = condition_data['fluxes']
    
    # Compute flux differential (fitted - reference)
    flux_differential = {}
    all_rxns = set(fitted_flux.keys()) | set(reference_flux.keys())
    
    for rxn_id in all_rxns:
        fitted_val = fitted_flux.get(rxn_id, 0)
        ref_val = reference_flux.get(rxn_id, 0)
        diff = fitted_val - ref_val
        flux_differential[rxn_id] = diff
    
    # Statistics on differential
    positive_changes = sum(1 for v in flux_differential.values() if v > 0.01)
    negative_changes = sum(1 for v in flux_differential.values() if v < -0.01)
    unchanged = len(flux_differential) - positive_changes - negative_changes
    
    print(f"\nFlux differential statistics:")
    print(f"  Increased (>0.01): {positive_changes} reactions")
    print(f"  Decreased (<-0.01): {negative_changes} reactions")
    print(f"  Unchanged: {unchanged} reactions")
    
    # Show top 10 largest changes
    sorted_diffs = sorted(flux_differential.items(), key=lambda x: abs(x[1]), reverse=True)
    print(f"\nTop 10 largest flux changes:")
    for rxn_id, diff in sorted_diffs[:10]:
        direction = "+" if diff > 0 else ""
        print(f"  {rxn_id}: {direction}{diff:.4f}")
    
    # Load model for metadata
    model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")
    
    # Create Escher map visualization for flux differential
    os.makedirs(util.output_dir, exist_ok=True)
    output_file = util.create_map_html2(
        model=model.model,
        flux=flux_differential,
        map="full",
        output_path=f"{util.output_dir}/escher_berdl_flux_diff_{short_name}.html",
    )
    
    print(f"\nFlux differential map saved to: {output_file}")
    
    # Display link
    from IPython.display import HTML, display
    display(HTML(f'<a href="{output_file}" target="_blank">Open Flux Differential Map ({short_name})</a>'))

2026-02-21 09:32:06,548 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-21 09:32:06,548 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-21 09:32:06,548 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-21 09:32:06,549 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


/Users/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-21 09:32:06,916 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-02-21 09:32:06,918 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-02-21 09:32:06,919 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-21 09:32:06,919 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


/Users/chenry/.npm-global/bin/claude


2026-02-21 09:32:07,131 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


Computing flux differential for: ACN2586

Flux differential statistics:
  Increased (>0.01): 143 reactions
  Decreased (<-0.01): 186 reactions
  Unchanged: 1175 reactions

Top 10 largest flux changes:
  rxn05209_c0: -32.5386
  rxn05313_c0: +12.4699
  rxn05312_c0: -12.0130
  rxn08173_c0: -9.8310
  EX_cpd00020_e0: +9.1011
  rxn05469_c0: -9.1011
  rxn00171_c0: +7.9639
  rxn08062_c0: -7.4724
  rxn00154_c0: -7.1847
  rxn08032_c0: -6.8958


2026-02-21 09:32:07,490 - __main__.NotebookUtil - INFO - Loaded map 'full' from local index: /Users/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-02-21 09:32:07,495 - __main__.NotebookUtil - INFO - Updated names for 759 reactions in map



Flux differential map saved to: None


## Flux Differential Dictionary for Manual Escher Rendering

This step:
1. Computes the flux differential (fitted - reference) for a selected condition
2. Filters to only non-trivial changes (|diff| > 0.01)
3. Prints the dictionary in a format suitable for copy-pasting into Escher's reaction data input

In [1]:
%run util.py

import math

# Load data
results = util.load("ADP1BERDLFoldChangeAnalysis")
reference_flux = util.load("berdl_fc_reference_flux")

# Select condition (change this to view different conditions)
selected_condition = "Pyruvate_ACN2586_DgoA_2025"
short_name = selected_condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')

print(f"Flux differential dictionary for: {short_name}")
print(f"Format: reaction_id -> (fitted - reference) flux value")
print(f"Only showing |diff| > 0.01")
print("=" * 60)

condition_data = results[selected_condition]
fitted_flux = condition_data['fluxes']

# Compute flux differential and filter
flux_diff_dict = {}
all_rxns = set(fitted_flux.keys()) | set(reference_flux.keys())
for rxn_id in sorted(all_rxns):
    fitted_val = abs(fitted_flux.get(rxn_id, 0))
    ref_val = abs(reference_flux.get(rxn_id, 0))
    if ref_val > 0.000001 and fitted_val > 0.000001:
        diff = math.log2(fitted_val/ref_val)
        if diff > 1:
            diff = 1
        if diff < -1:
            diff = -1
        diff = diff + 1
        flux_diff_dict[rxn_id] = diff
        flux_diff_dict[rxn_id+"-rev"] = diff
        
print(f"\n{len(flux_diff_dict)} reactions with non-trivial flux changes:\n")

print(f"\nTotal: {len(flux_diff_dict)} reactions")
print(f"Increased: {sum(1 for v in flux_diff_dict.values() if v > 0)}")
print(f"Decreased: {sum(1 for v in flux_diff_dict.values() if v < 0)}")
util.save("flux_diff",flux_diff_dict)

/Users/chenry/Dropbox/Projects/KBUtilLib/src


[KBUtilLib] Failed to import rcsb_pdb_utils: ModuleNotFoundError: No module named 'aiohttp'


modelseedpy 0.4.2


2026-02-21 14:34:15,194 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-21 14:34:15,195 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-21 14:34:15,196 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


loading biochemistry database from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


2026-02-21 14:34:20,273 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase
2026-02-21 14:34:20,680 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2026-02-21 14:34:20,682 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-02-21 14:34:20,682 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-21 14:34:20,682 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


cobrakbase 0.4.0
/Users/chenry/.npm-global/bin/claude


2026-02-21 14:34:20,935 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: claude-code


Flux differential dictionary for: ACN2586
Format: reaction_id -> (fitted - reference) flux value
Only showing |diff| > 0.01

816 reactions with non-trivial flux changes:


Total: 816 reactions
Increased: 802
Decreased: 0


## Generate All Condition Maps

This step:
1. Generates Escher maps for all conditions in a single batch
2. Creates fitted flux maps and flux differential maps for each strain
3. Provides links to all generated visualizations

In [4]:
%run util.py

# Load all data
results = util.load("ADP1BERDLFoldChangeAnalysis")
reference_flux = util.load("berdl_fc_reference_flux")

# Load model once for all visualizations
model = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")

print("Generating Escher maps for all conditions...")
print("=" * 80)

generated_files = {
    'reference': None,
    'fitted': {},
    'differential': {}
}

os.makedirs(util.output_dir, exist_ok=True)

# Generate reference flux map first
print("\n[Reference Flux - BERDL]")
ref_output = util.create_map_html2(
    model=model.model,
    flux=reference_flux,
    map="full",
    output_path=f"{util.output_dir}/escher_berdl_reference_flux.html",
)
generated_files['reference'] = ref_output
print(f"  Saved: {ref_output}")

# Generate maps for each condition
for condition, data in results.items():
    short_name = condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')
    print(f"\n[{short_name}]")
    
    if data.get('status') == 'error':
        print(f"  SKIPPED - Error in analysis")
        continue
    
    fitted_flux = data['fluxes']
    
    # Generate fitted flux map
    fitted_output = util.create_map_html2(
        model=model.model,
        flux=fitted_flux,
        map="full",
        output_path=f"{util.output_dir}/escher_berdl_fitted_flux_{short_name}.html"
    )
    generated_files['fitted'][short_name] = fitted_output
    print(f"  Fitted flux: {fitted_output}")
    
    # Compute and generate differential map
    flux_differential = {}
    all_rxns = set(fitted_flux.keys()) | set(reference_flux.keys())
    for rxn_id in all_rxns:
        diff = fitted_flux.get(rxn_id, 0) - reference_flux.get(rxn_id, 0)
        flux_differential[rxn_id] = diff
    
    diff_output = util.create_map_html2(
        model=model.model,
        flux=flux_differential,
        map="full",
        output_path=f"{util.output_dir}/escher_berdl_flux_diff_{short_name}.html"
    )
    generated_files['differential'][short_name] = diff_output
    print(f"  Differential: {diff_output}")

# Save file list for reference
util.save("berdl_fold_change_escher_files", generated_files)

print("\n" + "=" * 80)
print("All maps generated successfully!")
print(f"Total files: 1 reference + {len(generated_files['fitted'])} fitted + {len(generated_files['differential'])} differential")

# Display links to all files
from IPython.display import HTML, display

html_links = "<h3>Generated Escher Maps (BERDL Reference)</h3>"
html_links += "<h4>Reference Flux:</h4>"
html_links += f'<p><a href="{generated_files["reference"]}" target="_blank">Reference Flux (BERDL Fitness-Constrained)</a></p>'

html_links += "<h4>Fitted Flux Maps:</h4><ul>"
for name, path in generated_files['fitted'].items():
    html_links += f'<li><a href="{path}" target="_blank">{name}</a></li>'
html_links += "</ul>"

html_links += "<h4>Flux Differential Maps (vs BERDL Reference):</h4><ul>"
for name, path in generated_files['differential'].items():
    html_links += f'<li><a href="{path}" target="_blank">{name}</a></li>'
html_links += "</ul>"

display(HTML(html_links))

2026-02-21 20:31:15,154 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-02-21 20:31:15,154 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-02-21 20:31:15,155 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-02-21 20:31:15,783 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-02-21 20:31:15,785 - __main__.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-02-21 20:31:15,786 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-21 20:31:15,787 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-02-21 20:31:15,803 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo
2026-02-21 20:31:16,550 - __main__.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-02-21 20:31:16,555 - __main__.NotebookUtil - INFO - Updated names for 759 reactions in map


Generating Escher maps for all conditions...

[Reference Flux - BERDL]


2026-02-21 20:31:17,565 - __main__.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-02-21 20:31:17,568 - __main__.NotebookUtil - INFO - Updated names for 759 reactions in map


...

  Fitted flux: /home/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_fitted_flux_ACN3429.html


2026-02-21 20:31:31,280 - __main__.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-02-21 20:31:31,284 - __main__.NotebookUtil - INFO - Updated names for 759 reactions in map


  Differential: /home/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_flux_diff_ACN3429.html

[ACN3430]


2026-02-21 20:31:32,321 - __main__.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-02-21 20:31:32,325 - __main__.NotebookUtil - INFO - Updated names for 759 reactions in map


  Fitted flux: /home/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_fitted_flux_ACN3430.html
  Differential: /home/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_flux_diff_ACN3430.html

All maps generated successfully!
Total files: 1 reference + 6 fitted + 6 differential
